In [1]:
import pandas as pd
import numpy as np
from bertopic import BERTopic

In [2]:
# numpy needs to have the correct version!!
import numpy as np
print("NumPy version:", np.__version__)


NumPy version: 1.26.4


In [3]:
# Load the cleaned data
df = pd.read_csv('../outputs/cleaned_text_data.csv')

# For Option2 texts
option2_docs = df[df['religious_group'] == 'option2']['cleaned_content'].dropna()
option2_docs = option2_docs[option2_docs.str.strip() != '']
option2_docs = option2_docs.tolist()

# For Option1 texts
option1_docs = df[df['religious_group'] == 'option1']['cleaned_content'].dropna()
option1_docs = option1_docs[option1_docs.str.strip() != '']
option1_docs = option1_docs.tolist()

print(f"Option2 docs: {len(option2_docs)}")
print(f"Option1 docs: {len(option1_docs)}")

Option2 docs: 3104
Option1 docs: 2197


In [4]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer_model = CountVectorizer(stop_words="english", min_df=2, ngram_range=(1, 2))

In [5]:
import openai
from bertopic.representation import OpenAI
from dotenv import load_dotenv
import os

# Fine-tune topic representations with GPT
load_dotenv()
client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
openai_model = OpenAI(client, model="gpt-4o-mini", chat=True)

In [6]:
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, OpenAI

# KeyBERT
keybert_model = KeyBERTInspired()

# MMR
mmr_model = MaximalMarginalRelevance(diversity=0.3)


# All representation models
representation_model = {
    "KeyBERT": keybert_model,
    "OpenAI": openai_model,  # Uncomment if you will use OpenAI
    "MMR": mmr_model,
}

In [7]:
# OPTIMIZED BERTOPIC -  faster processing
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN

print("Setting up optimized BERTopic models...")

# Use a faster, lighter embedding model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Reduce UMAP dimensions for faster processing
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)

# Optimize HDBSCAN for speed
hdbscan_model = HDBSCAN(min_cluster_size=10, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

# Create optimized BERTopic model for Option2
option2_topic_model_fast = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model, 
    hdbscan_model=hdbscan_model,
    language="english",
    calculate_probabilities=True,
    verbose=True,
    vectorizer_model=vectorizer_model,
    representation_model=representation_model
)

print(f"Processing {len(option2_docs)} Option2 documents...")
option2_topics, option2_probs = option2_topic_model_fast.fit_transform(option2_docs)
print("Option2 topic modeling complete.")


Setting up optimized BERTopic models...


2025-08-13 11:16:55,913 - BERTopic - Embedding - Transforming documents to embeddings.


Processing 3104 Option2 documents...


Batches:   0%|          | 0/97 [00:00<?, ?it/s]

2025-08-13 11:17:08,615 - BERTopic - Embedding - Completed ✓
2025-08-13 11:17:08,615 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
2025-08-13 11:17:20,771 - BERTopic - Dimensionality - Completed ✓
2025-08-13 11:17:20,772 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-08-13 11:17:21,056 - BERTopic - Cluster - Completed ✓
2025-08-13 11:17:21,060 - BERTopic - Representation - Fine-tuning topics using representation models.
100%|██████████| 52/52 [00:41<00:00,  1.24it/s]
2025-08-13 11:18:10,773 - BERTopic - Representation - Completed ✓


Option2 topic modeling complete.


In [8]:
# Create optimized BERTopic model for Option1
option1_topic_model_fast = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model, 
    hdbscan_model=hdbscan_model,
    language="english",
    calculate_probabilities=True,
    verbose=True,
    vectorizer_model=vectorizer_model,
    representation_model=representation_model
)

print(f"Processing {len(option1_docs)} Option1 documents...")
option1_topics, option1_probs = option1_topic_model_fast.fit_transform(option1_docs)
print("Option1 topic modeling complete.")


2025-08-13 11:18:13,436 - BERTopic - Embedding - Transforming documents to embeddings.


Processing 2197 Option1 documents...


Batches:   0%|          | 0/69 [00:00<?, ?it/s]

2025-08-13 11:18:19,903 - BERTopic - Embedding - Completed ✓
2025-08-13 11:18:19,903 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-08-13 11:18:25,391 - BERTopic - Dimensionality - Completed ✓
2025-08-13 11:18:25,392 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-08-13 11:18:25,549 - BERTopic - Cluster - Completed ✓
2025-08-13 11:18:25,551 - BERTopic - Representation - Fine-tuning topics using representation models.
100%|██████████| 46/46 [00:27<00:00,  1.69it/s]
2025-08-13 11:18:57,595 - BERTopic - Representation - Completed ✓


Option1 topic modeling complete.


In [10]:
# save raw model
option1_topic_model_fast.save('../outputs/models/option1_topic_model_fast', serialization='safetensors')
option2_topic_model_fast.save('../outputs/models/option2_topic_model_fast', serialization='safetensors')

### additional details of the raw model

In [11]:
# Add top words as a column with proper error handling
def get_top_words(topic_num, topic_model):
    if topic_num == -1:  # Outlier topic
        return []
    topic_words = topic_model.get_topic(topic_num)
    if topic_words and isinstance(topic_words, list):
        return [word for word, score in topic_words[:10]]
    else:
        return []

In [17]:
# Create CSV files that EXACTLY match the topic modeling documents
import os

# Create output directory if it doesn't exist
os.makedirs('../outputs/analysis_results', exist_ok=True)

# IMPORTANT: Use the exact same filtering logic as Cell 3 to ensure perfect alignment
# Option2 - replicate exact filtering from Cell 3
option2_filtered_series = df[df['religious_group'] == 'option2']['cleaned_content'].dropna()
option2_filtered_series = option2_filtered_series[option2_filtered_series.str.strip() != '']
option2_valid_indices = option2_filtered_series.index  # Original indices in main df

# Option1 - replicate exact filtering from Cell 3
option1_filtered_series = df[df['religious_group'] == 'option1']['cleaned_content'].dropna()
option1_filtered_series = option1_filtered_series[option1_filtered_series.str.strip() != '']
option1_valid_indices = option1_filtered_series.index  # Original indices in main df

# Create dataframes using the EXACT same rows as topic modeling
option2_df = df.loc[option2_valid_indices].copy().reset_index(drop=True)
option1_df = df.loc[option1_valid_indices].copy().reset_index(drop=True)

# Save CSV files
option2_df.to_csv('../outputs/analysis_results/option2_documents.csv', index=False)
option1_df.to_csv('../outputs/analysis_results/option1_documents.csv', index=False)

In [12]:
# get raw content samples for each topic
option2_topic_info = option2_topic_model_fast.get_topic_info()

# Add top words as a column using lambda to pass the model
option2_topic_info['top_words'] = option2_topic_info['Topic'].apply(
    lambda topic_num: get_top_words(topic_num, option2_topic_model_fast)
)

def get_most_representative_samples(topic_num, topics, docs, probs, max_samples=3):
    """Get the most representative (highest confidence) text samples for a topic
    
    Returns:
        tuple: (samples, indices) where samples are the text content and indices are the original document indices
        For outlier topic (-1), returns empty lists
    """
    if topic_num == -1:  # Outlier topic - skip processing
        return [], []
    else:
        # Find all documents assigned to this topic
        topic_indices = [i for i, t in enumerate(topics) if t == topic_num]
        
        if len(topic_indices) == 0:
            return [], []
        
        # Get confidence scores for this topic
        topic_probs = [probs[i][topic_num] if topic_num < len(probs[i]) else 0 for i in topic_indices]
        
        # Get indices of most confident documents
        if len(topic_indices) <= max_samples:
            selected_indices = topic_indices
        else:
            # Sort by confidence and take top samples
            sorted_pairs = sorted(zip(topic_probs, topic_indices), reverse=True)
            selected_indices = [idx for _, idx in sorted_pairs[:max_samples]]
        
        samples = [docs[i] for i in selected_indices]
        return samples, selected_indices

# tuple return value for raw content samples and indices
def extract_samples_and_indices(topic_num):
    samples, indices = get_most_representative_samples(topic_num, option2_topics, option2_docs, option2_probs, max_samples=5)
    return samples

def extract_sample_indices(topic_num):
    samples, indices = get_most_representative_samples(topic_num, option2_topics, option2_docs, option2_probs, max_samples=5)
    return indices

option2_topic_info['raw_content_samples'] = option2_topic_info['Topic'].apply(extract_samples_and_indices)
option2_topic_info['sample_indices'] = option2_topic_info['Topic'].apply(extract_sample_indices)


In [13]:
# Save to CSV
option2_topic_info.to_csv('../outputs/analysis_results/option2_topic_info_detailed_raw.csv', index=False)
print(f"Option2 topic info saved. Found {len(option2_topic_info)} topics.")

Option2 topic info saved. Found 52 topics.


In [14]:

option1_topic_info = option1_topic_model_fast.get_topic_info()

# Add top words as a column using lambda to pass the model
option1_topic_info['top_words'] = option1_topic_info['Topic'].apply(
    lambda topic_num: get_top_words(topic_num, option1_topic_model_fast)
)

# Helper functions for Option1 
def extract_samples_and_indices_option1(topic_num):
    samples, indices = get_most_representative_samples(topic_num, option1_topics, option1_docs, option1_probs, max_samples=5)
    return samples

def extract_sample_indices_option1(topic_num):
    samples, indices = get_most_representative_samples(topic_num, option1_topics, option1_docs, option1_probs, max_samples=5)
    return indices

# Add raw content samples and their indices
option1_topic_info['raw_content_samples'] = option1_topic_info['Topic'].apply(extract_samples_and_indices_option1)
option1_topic_info['sample_indices'] = option1_topic_info['Topic'].apply(extract_sample_indices_option1)

In [15]:
# Save to CSV
option1_topic_info.to_csv('../outputs/analysis_results/option1_topic_info_detailed_raw.csv', index=False)
print(f"Option1 topic info saved. Found {len(option1_topic_info)} topics.")

Option1 topic info saved. Found 46 topics.


## fine-tune topics

### reduce outliers for option 1
#### NOTE!!! This step directly updates the raw model! (Not recommended).

In [20]:
# Reduce outliers for Option1 using BERTopic's reduce_outliers method
print("Reducing outliers for Option1...")
print(f"Before: {sum(1 for t in option1_topics if t == -1)} outliers out of {len(option1_topics)} documents")

# Use BERTopic's reduce_outliers method
new_topics_option1 = option1_topic_model_fast.reduce_outliers(option1_docs, option1_topics, 
                                                             probabilities=option1_probs,
                                                             threshold=0.05, 
                                                             strategy="probabilities")

# Update the model with new topic assignments
option1_topic_model_fast.update_topics(option1_docs, topics=new_topics_option1)

print(f"After: {sum(1 for t in new_topics_option1 if t == -1)} outliers out of {len(new_topics_option1)} documents")
print(f"Reduced outliers by {sum(1 for t in option1_topics if t == -1) - sum(1 for t in new_topics_option1 if t == -1)} documents")

2025-08-12 11:47:47,838 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Reducing outliers for Option1...
Before: 655 outliers out of 2197 documents
After: 253 outliers out of 2197 documents
Reduced outliers by 402 documents


In [25]:
# Save the updated Option1 topic information and representations (CSV format)
print("Saving updated Option1 topic information with representations...")

# Get the updated topic info from the model  
option1_topic_info_updated = option1_topic_model_fast.get_topic_info()

# Add top words as a column using lambda to pass the model
option1_topic_info_updated['top_words'] = option1_topic_info_updated['Topic'].apply(
    lambda topic_num: get_top_words(topic_num, option1_topic_model_fast)
)

# Helper functions for updated topics (using new_topics_option1)
def extract_samples_and_indices_updated_option1(topic_num):
    samples, indices = get_most_representative_samples(topic_num, new_topics_option1, option1_docs, option1_probs, max_samples=5)
    return samples

def extract_sample_indices_updated_option1(topic_num):
    samples, indices = get_most_representative_samples(topic_num, new_topics_option1, option1_docs, option1_probs, max_samples=5)
    return indices

# Add raw content samples and their indices (using UPDATED topic assignments)
option1_topic_info_updated['raw_content_samples'] = option1_topic_info_updated['Topic'].apply(extract_samples_and_indices_updated_option1)
option1_topic_info_updated['sample_indices'] = option1_topic_info_updated['Topic'].apply(extract_sample_indices_updated_option1)

# Save to CSV
option1_topic_info_updated.to_csv('../outputs/analysis_results/option1_topic_info_reduced_outliers.csv', index=False)


Saving updated Option1 topic information with representations...


In [22]:
# Initialize models for embedding generation and visualization
from sentence_transformers import SentenceTransformer
from umap import UMAP

# Use the same embedding model as BERTopic (for consistency)
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")

# Initialize UMAP reducer with same settings as earlier
reducer = UMAP(n_neighbors=10, n_components=2, min_dist=0.0, metric='cosine', random_state=42)

print("Models initialized: SentenceTransformer and UMAP ready for embedding generation")


Models initialized: SentenceTransformer and UMAP ready for embedding generation


In [23]:
# Generate embeddings and 2D reduction for Option1 documents
print("Encoding Option1 documents...")
option1_embeddings = sentence_model.encode(option1_docs, show_progress_bar=True)

print("Step 2: Reducing Option1 embeddings to 2D for fast visualization...")
# Use same reducer settings for consistency
option1_reduced_embeddings = reducer.fit_transform(option1_embeddings)

print(f"Option1: Generated {len(option1_embeddings)} embeddings and reduced to 2D")

Encoding Option1 documents...


Batches:   0%|          | 0/69 [00:00<?, ?it/s]

Step 2: Reducing Option1 embeddings to 2D for fast visualization...
Option1: Generated 2197 embeddings and reduced to 2D


In [24]:
# Create interactive DataMapPlots for UPDATED Option1 topics (after outlier reduction)
print("Step 3: Creating interactive DataMapPlots for updated Option1 topics...")

try:
    # Use the NEW topic assignments (after outlier reduction)
    option1_datamap_updated = option1_topic_model_fast.visualize_document_datamap(
        option1_docs,
        topics=new_topics_option1,  # Use the updated topic assignments!
        reduced_embeddings=option1_reduced_embeddings,
        interactive=True,
        title="Option1 Documents Topic Map (Updated - Reduced Outliers)"
    )
    
    # Save the interactive plot
    option1_datamap_updated.save("../outputs/analysis_results/option1_interactive_datamap_reduced_outliers.html")
    
except Exception as e:
    print(f"Error creating Option1 updated DataMapPlot: {e}")

print(f"Updated Option1 topics: {len(set(new_topics_option1))} unique topics (including outliers)")
print(f"Outliers: {sum(1 for t in new_topics_option1 if t == -1)} documents")


Step 3: Creating interactive DataMapPlots for updated Option1 topics...
Updated Option1 topics: 46 unique topics (including outliers)
Outliers: 253 documents


### reduce outliers for option 2


In [26]:
# Reduce outliers for Option2 using BERTopic's reduce_outliers method
print("Reducing outliers for Option2...")
print(f"Before: {sum(1 for t in option2_topics if t == -1)} outliers out of {len(option2_topics)} documents")

# Use BERTopic's reduce_outliers method
new_topics_option2 = option2_topic_model_fast.reduce_outliers(option2_docs, option2_topics, 
                                                             probabilities=option2_probs,
                                                             threshold=0.05, 
                                                             strategy="probabilities")

# Update the model with new topic assignments
option2_topic_model_fast.update_topics(option2_docs, topics=new_topics_option2)

print(f"After: {sum(1 for t in new_topics_option2 if t == -1)} outliers out of {len(new_topics_option2)} documents")
print(f"Reduced outliers by {sum(1 for t in option2_topics if t == -1) - sum(1 for t in new_topics_option2 if t == -1)} documents")


2025-08-12 12:05:38,747 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Reducing outliers for Option2...
Before: 857 outliers out of 3104 documents
After: 456 outliers out of 3104 documents
Reduced outliers by 401 documents


In [27]:
# Save the updated Option2 topic information and representations (CSV format)
print("Saving updated Option2 topic information with representations...")

# Get the updated topic info from the model  
option2_topic_info_updated = option2_topic_model_fast.get_topic_info()

# Add top words as a column using lambda to pass the model
option2_topic_info_updated['top_words'] = option2_topic_info_updated['Topic'].apply(
    lambda topic_num: get_top_words(topic_num, option2_topic_model_fast)
)

# Helper functions for updated topics (using new_topics_option2)
def extract_samples_and_indices_updated_option2(topic_num):
    samples, indices = get_most_representative_samples(topic_num, new_topics_option2, option2_docs, option2_probs, max_samples=5)
    return samples

def extract_sample_indices_updated_option2(topic_num):
    samples, indices = get_most_representative_samples(topic_num, new_topics_option2, option2_docs, option2_probs, max_samples=5)
    return indices

# Add raw content samples and their indices (using UPDATED topic assignments)
option2_topic_info_updated['raw_content_samples'] = option2_topic_info_updated['Topic'].apply(extract_samples_and_indices_updated_option2)
option2_topic_info_updated['sample_indices'] = option2_topic_info_updated['Topic'].apply(extract_sample_indices_updated_option2)

# Save to CSV
option2_topic_info_updated.to_csv('../outputs/analysis_results/option2_topic_info_reduced_outliers.csv', index=False)


Saving updated Option2 topic information with representations...


In [29]:
# Generate embeddings and 2D reduction for Option2 documents
print("Encoding Option2 documents...")
option2_embeddings = sentence_model.encode(option2_docs, show_progress_bar=True)

print("Reducing Option2 embeddings to 2D for fast visualization...")
# Use same reducer settings for consistency (fit a new reducer for Option2)
option2_reducer = UMAP(n_neighbors=10, n_components=2, min_dist=0.0, metric='cosine', random_state=42)
option2_reduced_embeddings = option2_reducer.fit_transform(option2_embeddings)

print(f"Option2: Generated {len(option2_embeddings)} embeddings and reduced to 2D")

Encoding Option2 documents...


Batches:   0%|          | 0/97 [00:00<?, ?it/s]

Reducing Option2 embeddings to 2D for fast visualization...
Option2: Generated 3104 embeddings and reduced to 2D


In [30]:
# Create interactive DataMapPlots for UPDATED Option2 topics (after outlier reduction)
print("Creating interactive DataMapPlots for updated Option2 topics...")

try:
    # Use the NEW topic assignments (after outlier reduction)
    option2_datamap_updated = option2_topic_model_fast.visualize_document_datamap(
        option2_docs,
        topics=new_topics_option2,  # Use the updated topic assignments!
        reduced_embeddings=option2_reduced_embeddings,
        interactive=True,
        title="Option2 Documents Topic Map (Updated - Reduced Outliers)"
    )
    
    # Save the interactive plot
    option2_datamap_updated.save("../outputs/analysis_results/option2_interactive_datamap_reduced_outliers.html")
    print("Option2 updated DataMapPlot saved to option2_interactive_datamap_reduced_outliers.html")
    
except Exception as e:
    print(f"Error creating Option2 updated DataMapPlot: {e}")



Creating interactive DataMapPlots for updated Option2 topics...
Option2 updated DataMapPlot saved to option2_interactive_datamap_reduced_outliers.html


### removing teacher interviews

In [31]:
# Filter out interviews using file_type column 
print("Filtering out interview data for textbook-only analysis (using file_type)...")

# Load the Option1 and Option2 dataframes 
option1_df = pd.read_csv('../outputs/analysis_results/option1_documents.csv')
option2_df = pd.read_csv('../outputs/analysis_results/option2_documents.csv')

# Filter out interviews using the file_type column (much simpler!)
option1_textbooks = option1_df[option1_df['file_type'] != 'interview'].copy()
option2_textbooks = option2_df[option2_df['file_type'] != 'interview'].copy()

# Extract cleaned content for topic modeling
option1_docs_textbooks = option1_textbooks['cleaned_content'].tolist()
option2_docs_textbooks = option2_textbooks['cleaned_content'].tolist()

# Save the filtered dataframes for reference
option1_textbooks.to_csv('../outputs/analysis_results/option1_no_interviews.csv', index=False)
option2_textbooks.to_csv('../outputs/analysis_results/option2_no_interviews.csv', index=False)

Filtering out interview data for textbook-only analysis (using file_type)...


In [34]:
# Train Option1 textbook-only topic model
print("Training Option1 textbook-only topic model...")

# Initialize components (same as before)
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=10, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
vectorizer_model = CountVectorizer(stop_words="english", min_df=2, ngram_range=(1, 2))

# Representation models
keybert_model = KeyBERTInspired()
mmr_model = MaximalMarginalRelevance(diversity=0.3)
representation_model = {
    "KeyBERT": keybert_model,
    "MMR": mmr_model,
    "OpenAI": openai_model
}

# Create Option1 textbook-only model
option1_textbook_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    language="english",
    calculate_probabilities=True,
    verbose=True,
    vectorizer_model=vectorizer_model,
    representation_model=representation_model
)

option1_topics_textbooks, option1_probs_textbooks = option1_textbook_model.fit_transform(option1_docs_textbooks)


Training Option1 textbook-only topic model...


2025-08-12 15:10:23,079 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/59 [00:00<?, ?it/s]

2025-08-12 15:10:27,853 - BERTopic - Embedding - Completed ✓
2025-08-12 15:10:27,853 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-08-12 15:10:32,168 - BERTopic - Dimensionality - Completed ✓
2025-08-12 15:10:32,168 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-08-12 15:10:32,316 - BERTopic - Cluster - Completed ✓
2025-08-12 15:10:32,321 - BERTopic - Representation - Fine-tuning topics using representation models.
100%|██████████| 50/50 [00:34<00:00,  1.44it/s]
2025-08-12 15:11:11,944 - BERTopic - Representation - Completed ✓


In [39]:

# Get topic info and add representative samples with indices
option1_textbook_topic_info = option1_textbook_model.get_topic_info()

# Add top words column
option1_textbook_topic_info['top_words'] = option1_textbook_topic_info['Topic'].apply(
    lambda topic_num: get_top_words(topic_num, option1_textbook_model)
)

# Helper functions for textbook topics
def extract_samples_textbooks_option1(topic_num):
    samples, indices = get_most_representative_samples(topic_num, option1_topics_textbooks, option1_docs_textbooks, option1_probs_textbooks, max_samples=5)
    return samples

def extract_indices_textbooks_option1(topic_num):
    samples, indices = get_most_representative_samples(topic_num, option1_topics_textbooks, option1_docs_textbooks, option1_probs_textbooks, max_samples=5)
    return indices

# Add representative samples and indices
option1_textbook_topic_info['raw_content_samples'] = option1_textbook_topic_info['Topic'].apply(extract_samples_textbooks_option1)
option1_textbook_topic_info['sample_indices'] = option1_textbook_topic_info['Topic'].apply(extract_indices_textbooks_option1)

# Save to CSV
option1_textbook_topic_info.to_csv('../outputs/analysis_results/option1_topic_info_no_interviews.csv', index=False)


In [38]:
# Train Option2 textbook-only topic model
print("Training Option2 textbook-only topic model...")

# Create new BERTopic model for Option2 textbooks only (using same settings)
option2_textbook_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    language="english",
    calculate_probabilities=True,
    verbose=True,
    vectorizer_model=vectorizer_model,
    representation_model=representation_model
)

option2_topics_textbooks, option2_probs_textbooks = option2_textbook_model.fit_transform(option2_docs_textbooks)

# Display topic info
option2_textbook_topic_info = option2_textbook_model.get_topic_info()

option2_textbook_topic_info.to_csv('../outputs/analysis_results/option2_topic_info_no_interviews.csv', index=False)

2025-08-12 15:13:02,461 - BERTopic - Embedding - Transforming documents to embeddings.


Training Option2 textbook-only topic model...


Batches:   0%|          | 0/88 [00:00<?, ?it/s]

2025-08-12 15:13:10,590 - BERTopic - Embedding - Completed ✓
2025-08-12 15:13:10,590 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-08-12 15:13:18,351 - BERTopic - Dimensionality - Completed ✓
2025-08-12 15:13:18,352 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-08-12 15:13:18,714 - BERTopic - Cluster - Completed ✓
2025-08-12 15:13:18,717 - BERTopic - Representation - Fine-tuning topics using representation models.
100%|██████████| 53/53 [00:30<00:00,  1.75it/s]
2025-08-12 15:13:54,074 - BERTopic - Representation - Completed ✓


In [40]:
# Get topic info and add representative samples with indices
option2_textbook_topic_info = option2_textbook_model.get_topic_info()

# Add top words column
option2_textbook_topic_info['top_words'] = option2_textbook_topic_info['Topic'].apply(
    lambda topic_num: get_top_words(topic_num, option2_textbook_model)
)

# Helper functions for textbook topics
def extract_samples_textbooks_option2(topic_num):
    samples, indices = get_most_representative_samples(topic_num, option2_topics_textbooks, option2_docs_textbooks, option2_probs_textbooks, max_samples=5)
    return samples

def extract_indices_textbooks_option2(topic_num):
    samples, indices = get_most_representative_samples(topic_num, option2_topics_textbooks, option2_docs_textbooks, option2_probs_textbooks, max_samples=5)
    return indices

# Add representative samples and indices
option2_textbook_topic_info['raw_content_samples'] = option2_textbook_topic_info['Topic'].apply(extract_samples_textbooks_option2)
option2_textbook_topic_info['sample_indices'] = option2_textbook_topic_info['Topic'].apply(extract_indices_textbooks_option2)

# Save to CSV
option2_textbook_topic_info.to_csv('../outputs/analysis_results/option2_topic_info_no_interviews.csv', index=False)

### reduce number of topics

#### option2

In [26]:
# reduce number of topics
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN

print("Setting up optimized BERTopic models...")

# Use a faster, lighter embedding model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Reduce UMAP dimensions for faster processing
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)

# Optimize HDBSCAN for speed
hdbscan_model = HDBSCAN(min_cluster_size=15, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

Setting up optimized BERTopic models...


In [ ]:
# Create optimized BERTopic model for Option2
option2_topic_model_new = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model, 
    hdbscan_model=hdbscan_model,
    language="english",
    calculate_probabilities=True,
    verbose=True,
    vectorizer_model=vectorizer_model,
    representation_model=representation_model
)

print(f"Processing {len(option2_docs)} Option2 documents...")
option2_topics, option2_probs = option2_topic_model_new.fit_transform(option2_docs)
print("Option2 topic modeling with reduced number of topics complete.")


In [19]:
option2_topic_model_new.get_topic_info()

,Topic,Count,Name,Representation,KeyBERT,OpenAI,MMR,Representative_Docs
0,-1,435,-1_agreement_party_activity_archive,"[agreement, party, activity, archive, friday, ...","[friday agreement, agreement, activity, key fe...",[Friday Agreement Activities],"[agreement, party, friday agreement, rté, gove...",[friday agreement film next activity write new...
1,0,1099,0_ira_march_government_neill,"[ira, march, government, neill, civil, right, ...","[unionist, loyalist, protest, nationalist, civ...",[Northern Irish conflict],"[ira, march, neill, nationalist, civil right, ...",[ic nicra feb opposes speech nationalist react...
2,1,685,1_agreement_unionist_irish_government,"[agreement, unionist, irish, government, party...","[ulster unionist, unionist party, irish govern...",[Northern Ireland peace agreement],"[agreement, unionist, irish, féin, sinn féin, ...",[er joint declaration british irish government...
3,2,348,2_yeah_thing_kind_teaching,"[yeah, thing, kind, teaching, suppose, parent,...","[teach, teaching, taught, curriculum, social m...",[Teaching and Learning Dynamics],"[teaching, teach, kid, example, topic, thinkin...",[wear advertised k kind hard break role ok som...
4,3,165,3_hunger_strike_hunger strike_prisoner,"[hunger, strike, hunger strike, prisoner, sand...","[hunger strike, support hunger, hunger striker...",[Republican Hunger Strikes],"[hunger, hunger strike, hunger striker, protes...",[wearing clothes operating commander hunger st...
5,4,124,4_war_britain_german_éire,"[war, britain, german, éire, people, germany, ...","[war britain, world war, britain, war, treaty,...",[Ireland's Neutrality in War],"[britain, german, éire, free state, hitler, ir...",[peace war neutrality britain éire norman john...
6,5,103,5_agreement_friday agreement_friday_good friday,"[agreement, friday agreement, friday, good fri...","[friday agreement, agreement brought, agreemen...",[Good Friday Agreement],"[agreement, friday agreement, document, polici...",[friday agreement brought change policing ni t...
7,6,85,6_image_work_information_sourced,"[image, work, information, sourced, tool, inte...","[use image, image information, exporting, docu...",[Sourcing and Saving Images],"[task, filename, saved document, dedicated fol...",[n image information sourced internet designed...
8,7,60,7_information_lesson_key information_key,"[information, lesson, key information, key, di...","[learn interact, lesson plan, knowledge encour...",[Collaborative Learning Activities],"[key information, activity, topic, meet learni...",[nowledge active learning activity pair resear...


In [31]:
option2_topic_info_new = option2_topic_model_new.get_topic_info()
option2_topic_info_new.to_csv('../outputs/analysis_results/reduce_n_topics/option2/option2_topic_info_reduced_n_topics.csv', index=False)

#### option1

In [27]:
option1_topic_model_new = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model, 
    hdbscan_model=hdbscan_model,
    language="english",
    calculate_probabilities=True,
    verbose=True,
    vectorizer_model=vectorizer_model,
    representation_model=representation_model
)

print(f"Processing {len(option1_docs)} Option1 documents...")
option1_topics, option1_probs = option1_topic_model_new.fit_transform(option1_docs)
print("Option1 topic modeling with reduced number of topics complete.")

2025-08-13 11:50:12,317 - BERTopic - Embedding - Transforming documents to embeddings.


Processing 2197 Option1 documents...


Batches:   0%|          | 0/69 [00:00<?, ?it/s]

2025-08-13 11:50:17,068 - BERTopic - Embedding - Completed ✓
2025-08-13 11:50:17,069 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-08-13 11:50:22,375 - BERTopic - Dimensionality - Completed ✓
2025-08-13 11:50:22,376 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-08-13 11:50:22,454 - BERTopic - Cluster - Completed ✓
2025-08-13 11:50:22,455 - BERTopic - Representation - Fine-tuning topics using representation models.
100%|██████████| 10/10 [00:05<00:00,  1.73it/s]
2025-08-13 11:50:29,555 - BERTopic - Representation - Completed ✓


Option1 topic modeling with reduced number of topics complete.


In [28]:
option1_topic_model_new.get_topic_info()

,Topic,Count,Name,Representation,KeyBERT,OpenAI,MMR,Representative_Docs
0,-1,73,-1_library_software_archive_website,"[library, software, archive, website, image, s...","[audio, podcasts, music, radio, library, text,...",[Digital Archive Resources],"[library, archive, website, design, national a...",[earch icon illustration magnifying glass sear...
1,0,1542,0_irish_war_government_british,"[irish, war, government, british, state, brita...","[irish free, ulster, irish, unionist, independ...",[Irish Free State],"[irish, government, britain, belfast, ira, fre...",[madden mcbride ccea gcse chapter page peace w...
2,1,422,1_thing_yeah_teaching_lot,"[thing, yeah, teaching, lot, actually, people,...","[topic, suggestion teaching, teaching, discuss...",[Teaching controversial issues],"[teaching, lot, people, class, learning, teach...",[look use law source try parallel story time k...
3,2,35,2_tag_lesson_postwar_victorian,"[tag, lesson, postwar, victorian, workshop, fo...","[tag victorian, topic interwar, topic, concent...",[Historical Education Workshops],"[lesson, victorian, themed, collection, tag fo...",[n focussed topic session teach black victoria...
4,3,28,3_exceeded caused_exceeded_nodename_nodename s...,"[exceeded caused, exceeded, nodename, nodename...","[httpconnectionpool max, forbidden httpconnect...",[Network Resolution Errors],"[nodename, nodename servname, errno nodename, ...",[httpconnectionpool max retries exceeded cause...
5,4,23,4_icon_icon illustration_illustration_wayback,"[icon, icon illustration, illustration, waybac...","[wayback machine, machine text, text icon, win...",[Internet Archive Illustrations],"[icon, icon illustration, text, heart shape, i...",[tration computer application window wayback m...
6,5,20,5_debate_committee_oireachtas_dáil,"[debate, committee, oireachtas, dáil, parliame...","[debate dáil, dáil éireann, vote dáil, dáil de...",[Parliamentary processes and debates],"[debate, committee, oireachtas, parliament, sc...",[laid international parliamentary relation fre...
7,6,19,6_churchill_donate_page_sidebar hide,"[churchill, donate, page, sidebar hide, hide, ...","[community portal, sidebar, menu, contribute, ...",[Wikipedia navigation issues],"[churchill, page, sidebar hide, hide, sidebar,...",[nodename servname provided known county wikip...
8,7,19,7_website_criterion_cooky_success,"[website, criterion, cooky, success, accessibi...","[website cooky, website search, search website...",[Website Accessibility Services],"[criterion, accessibility, discovery, website ...",[website us cooky place essential cooky device...
9,8,16,8_exceeded caused_retries exceeded_max_max ret...,"[exceeded caused, retries exceeded, max, max r...","[caused nameresolutionerror, nameresolutionerr...",[Technical Errors],"[exceeded caused, retries exceeded, nameresolu...",[pyright right reserved privacy policy httpcon...


In [30]:
option1_topic_info_new = option1_topic_model_new.get_topic_info()
option1_topic_info_new.to_csv('../outputs/analysis_results/reduce_n_topics/option1/option1_topic_info_reduced_n_topics.csv', index=False)

# visualizations

## topics

In [14]:
fig1 = option1_topic_model_fast.visualize_topics()
fig1.show()

In [15]:
fig1.write_html("../outputs/analysis_results/option1_topics_visualization.html")

In [ ]:
# Save textbook topic models (handling pickle issues)
print("💾 Saving textbook topic models...")

import os
import pickle
import numpy as np

# Create models directory
os.makedirs('../outputs/analysis_results/models', exist_ok=True)

# Method 1: Use BERTopic's built-in save method (recommended)
try:
    print("🔧 Saving Option1 textbook model using BERTopic.save()...")
    option1_textbook_model.save("../outputs/analysis_results/models/option1_textbook_model", 
                               serialization="safetensors",
                               save_ctfidf=True,
                               save_embedding_model=True)
    print("✅ Option1 textbook model saved successfully")
except Exception as e:
    print(f"❌ Error saving Option1 model: {e}")

try:
    print("🔧 Saving Option2 textbook model using BERTopic.save()...")
    option2_textbook_model.save("../outputs/analysis_results/models/option2_textbook_model", 
                               serialization="safetensors",
                               save_ctfidf=True,
                               save_embedding_model=True)
    print("✅ Option2 textbook model saved successfully")
except Exception as e:
    print(f"❌ Error saving Option2 model: {e}")

# Method 2: Save essential components separately (fallback)
print("\n💾 Saving essential model components separately...")

# Save topic assignments and probabilities
np.save('../outputs/analysis_results/models/option1_textbook_topics.npy', option1_topics_textbooks)
np.save('../outputs/analysis_results/models/option1_textbook_probs.npy', option1_probs_textbooks)
np.save('../outputs/analysis_results/models/option2_textbook_topics.npy', option2_topics_textbooks)
np.save('../outputs/analysis_results/models/option2_textbook_probs.npy', option2_probs_textbooks)

# Save topic info (already saved as CSV but also as pickle)
with open('../outputs/analysis_results/models/option1_textbook_topic_info.pkl', 'wb') as f:
    pickle.dump(option1_textbook_topic_info, f)

with open('../outputs/analysis_results/models/option2_textbook_topic_info.pkl', 'wb') as f:
    pickle.dump(option2_textbook_topic_info, f)

# Save model parameters (for reconstruction)
model_params = {
    'umap_params': {
        'n_neighbors': 15,
        'n_components': 5,
        'min_dist': 0.0,
        'metric': 'cosine',
        'random_state_option1': 42,
        'random_state_option2': 43
    },
    'hdbscan_params': {
        'min_cluster_size': 10,
        'metric': 'euclidean',
        'cluster_selection_method': 'eom'
    },
    'vectorizer_params': {
        'stop_words': 'english',
        'min_df': 2,
        'ngram_range': (1, 2)
    },
    'embedding_model': 'all-MiniLM-L6-v2',
    'seeds': {'option1': 42, 'option2': 43}
}

with open('../outputs/analysis_results/models/textbook_model_params.pkl', 'wb') as f:
    pickle.dump(model_params, f)

print("✅ Essential components saved:")
print("   - Topic assignments: .npy files")
print("   - Topic info: .pkl files") 
print("   - Model parameters: textbook_model_params.pkl")
print("   - Full models: BERTopic .safetensors format")

print(f"\n📊 Summary:")
print(f"   Option1: {len(set(option1_topics_textbooks))} topics, {len(option1_docs_textbooks)} documents")
print(f"   Option2: {len(set(option2_topics_textbooks))} topics, {len(option2_docs_textbooks)} documents")


In [16]:
fig2 = option2_topic_model_fast.visualize_topics()
fig2.show()

In [ ]:
fig2.write_html('../outputs/analysis_results/option2_topic_model_visualization.html')

In [18]:
# Option1 heatmap with clustering
print("Creating Option1 topic similarity heatmap...")
option1_heatmap = option1_topic_model_fast.visualize_heatmap(
    n_clusters=4,  # Number of topic clusters to form
)
option1_heatmap.show()


Creating Option1 topic similarity heatmap...


In [19]:
# Save as HTML
option1_heatmap.write_html("../outputs/analysis_results/option1_topic_heatmap.html")

In [20]:
# Create topic similarity heatmaps with clustering

# Option2 heatmap with clustering
print("Creating Option2 topic similarity heatmap...")
option2_heatmap = option2_topic_model_fast.visualize_heatmap(
    n_clusters=5,  # Number of topic clusters to form
)
option2_heatmap.show()


Creating Option2 topic similarity heatmap...


In [28]:
# Save as HTML
option2_heatmap.write_html("../outputs/analysis_results/option2_topic_heatmap.html")

## documents

In [22]:
# OPTIMIZED Interactive Document Visualizations
# Following BERTopic documentation pipeline for speed optimization

print("Creating FAST interactive document visualizations...")
print("Step 1: Generating embeddings for documents...")

from sentence_transformers import SentenceTransformer
from umap import UMAP

# Use the same embedding model as our BERTopic (for consistency)
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")

# Generate embeddings for Option2 documents  
print("Encoding Option2 documents...")
option2_embeddings = sentence_model.encode(option2_docs, show_progress_bar=True)

print("Step 2: Reducing embeddings to 2D for fast visualization...")
# Pre-reduce embeddings to 2D (much faster for iterative visualization)
reducer = UMAP(n_neighbors=10, n_components=2, min_dist=0.0, metric='cosine', random_state=42)
option2_reduced_embeddings = reducer.fit_transform(option2_embeddings)

print(f"Option2: Generated {len(option2_embeddings)} embeddings and reduced to 2D")


Creating FAST interactive document visualizations...
Step 1: Generating embeddings for documents...
Encoding Option2 documents...


Batches:   0%|          | 0/97 [00:00<?, ?it/s]

Step 2: Reducing embeddings to 2D for fast visualization...
Option2: Generated 3104 embeddings and reduced to 2D


In [23]:
# Generate embeddings for Option1 documents
print("Encoding Option1 documents...")
option1_embeddings = sentence_model.encode(option1_docs, show_progress_bar=True)

print("Reducing Option1 embeddings to 2D...")
# Use the same reducer configuration for consistency
reducer1 = UMAP(n_neighbors=10, n_components=2, min_dist=0.0, metric='cosine', random_state=42)
option1_reduced_embeddings = reducer1.fit_transform(option1_embeddings)

print(f"Option1: Generated {len(option1_embeddings)} embeddings and reduced to 2D")

Encoding Option1 documents...


Batches:   0%|          | 0/69 [00:00<?, ?it/s]

Reducing Option1 embeddings to 2D...
Option1: Generated 2197 embeddings and reduced to 2D


In [ ]:
# Create FAST interactive DataMapPlots using pre-computed embeddings
print("Step 3: Creating interactive DataMapPlots with pre-computed embeddings...")

try:
    print("Creating Option2 interactive DataMapPlot...")
    # Use pre-computed reduced embeddings (MUCH faster!)
    option2_datamap = option2_topic_model_fast.visualize_document_datamap(
        option2_docs, 
        reduced_embeddings=option2_reduced_embeddings,  # Pre-computed!
        interactive=True,
        title="Option2 Documents Topic Map"
    )
    
    # Save as HTML file
    option2_datamap.save("../outputs/analysis_results/option2_interactive_datamap.html")
    print("Option2 interactive DataMapPlot saved.")
    
except Exception as e:
    print(f"Error creating Option2 DataMapPlot: {e}")

try:
    print("Creating Option1 interactive DataMapPlot...")
    option1_datamap = option1_topic_model_fast.visualize_document_datamap(
        option1_docs, 
        reduced_embeddings=option1_reduced_embeddings,  # Pre-computed!
        interactive=True,
        title="Option1 Documents Topic Map"
    )
    
    # Save as HTML file
    option1_datamap.save("../outputs/analysis_results/option1_interactive_datamap.html")
    print("Option1 interactive DataMapPlot saved.")
    
except Exception as e:
    print(f" Error creating Option1 DataMapPlot: {e}")


Step 3: Creating interactive DataMapPlots with pre-computed embeddings...
Creating Option2 interactive DataMapPlot...
Option2 interactive DataMapPlot saved.
Creating Option1 interactive DataMapPlot...
Option1 interactive DataMapPlot saved.


## hierarchy

In [33]:
option1_topic_model_fast.visualize_hierarchy()

In [34]:
option2_topic_model_fast.visualize_hierarchy()

In [36]:
hierarchical_topics_option1 = option1_topic_model_fast.hierarchical_topics(option1_docs)


100%|██████████| 44/44 [00:00<00:00, 611.95it/s]


In [48]:
# Visualize the hierarchical topics
print("Creating hierarchical topics visualization...")

# Use the hierarchical_model to create the visualization
hierarchy_fig_option1 = option1_topic_model_fast.visualize_hierarchy(hierarchical_topics=hierarchical_topics_option1)
hierarchy_fig_option1.show()

# Also save as HTML
hierarchy_fig_option1.write_html("../outputs/analysis_results/option1_hierarchical_topics.html")
print("Hierarchical topics visualization saved for option1")

Creating hierarchical topics visualization...


Hierarchical topics visualization saved for option1


there was a dimension mismatch when generating hierarchical topics for option 2 so a model was refit for this purpose. 

In [ ]:
# Create fresh model for hierarchical topics
print("Creating a fresh model for hierarchical topics...")

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN

# Create a simpler model specifically for hierarchical topics
simple_embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
simple_umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
simple_hdbscan_model = HDBSCAN(min_cluster_size=10, metric='euclidean', cluster_selection_method='eom')

hierarchical_model = BERTopic(
    embedding_model=simple_embedding_model,
    umap_model=simple_umap_model,
    hdbscan_model=simple_hdbscan_model,
    language="english",
    verbose=False  # Reduce output
)

print("Fitting fresh model for hierarchical analysis...")
hierarchical_model.fit(option2_docs)

print("Creating hierarchical topics with fresh model...")
hierarchical_topics_option2 = hierarchical_model.hierarchical_topics(option2_docs)
print("Option2 hierarchical topics created")
print(f"Shape: {hierarchical_topics_option2.shape}")


Creating a fresh model for hierarchical topics...
Fitting fresh model for hierarchical analysis...
Creating hierarchical topics with fresh model...


100%|██████████| 50/50 [00:00<00:00, 707.00it/s]

Option2 hierarchical topics created
Shape: (50, 8)


In [49]:
# Visualize the hierarchical topics
print("Creating hierarchical topics visualization...")

# Use the hierarchical_model to create the visualization
hierarchy_fig = hierarchical_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics_option2)
hierarchy_fig.show()

# Also save as HTML
hierarchy_fig.write_html("../outputs/analysis_results/option2_hierarchical_topics.html")
print("Hierarchical topics visualization saved for option2")


Creating hierarchical topics visualization...


Hierarchical topics visualization saved for option2
